## Tujuan

Tahap Modeling ini bertujuan untuk membangun model Machine Learning 
yang mampu memprediksi status kesejahteraan digital remaja 
(*Healthy*, *Moderate*, *At Risk*) berdasarkan pola perilaku digital 
dan indikator kesehatan mental mereka.

### Pendekatan: Classification dengan Random Forest Classifier

Pada proyek ini, kami memilih pendekatan **Supervised Learning** 
dengan algoritma **Random Forest Classifier**, bukan Unsupervised 
Learning (Clustering). Berikut alasannya:

**Mengapa Random Forest Classifier?**
- Data kita sudah memiliki label target yang jelas 
  (`digital_wellbeing_flag`: Healthy, Moderate, At Risk), sehingga 
  masalah ini adalah **klasifikasi**, bukan clustering.
- Random Forest tahan terhadap overfitting karena menggunakan 
  **ensemble** dari banyak decision tree.
- Mampu menangani fitur campuran (numerik & kategorikal yang sudah 
  di-encode).
- Memberikan informasi **feature importance** yang berguna untuk 
  interpretasi model.

**Bagaimana jika menggunakan Clustering?**

Jika kita menggunakan algoritma clustering seperti K-Means, hasilnya 
berupa **kelompok-kelompok (cluster)** tanpa label yang sudah 
ditentukan. Kita tidak bisa langsung mendapatkan prediksi 
*Healthy/Moderate/At Risk*, kita hanya tahu bahwa "remaja A dan B 
mirip satu sama lain", tapi tidak tahu kelompok mana yang *At Risk*. 
Selain itu, evaluasi clustering jauh lebih sulit karena tidak ada 
ground truth yang bisa dijadikan acuan. Karena data kita **sudah 
berlabel**, clustering bukan pilihan yang tepat.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

Langkah pertama adalah memuat dataset hasil tahap preprocessing (`preprocessed_data.csv`). Data ini sudah dalam skala yang seragam (StandardScaler) dan siap digunakan untuk modeling.

In [2]:
df = pd.read_csv('../data/preprocessed_data.csv')
print(f"[INFO] Data berhasil dimuat dengan {df.shape[0]} baris dan {df.shape[1]} kolom.")

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"[INFO] Jumlah Data Latih (Train): {X_train.shape[0]}")
print(f"[INFO] Jumlah Data Uji (Test): {X_test.shape[0]}")

[INFO] Data berhasil dimuat dengan 1200 baris dan 17 kolom.
[INFO] Jumlah Data Latih (Train): 960
[INFO] Jumlah Data Uji (Test): 240


### Train-Test Split (80:20)

Data dibagi dengan proporsi **80% Training** dan **20% Testing** 
menggunakan `stratify=y` untuk memastikan distribusi kelas target 
tetap proporsional di kedua subset.

In [3]:
# Inisialisasi & Training Model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

print("[INFO] Sedang melatih model Random Forest...")
rf_model.fit(X_train, y_train)
print("[INFO] Model berhasil dilatih!")

[INFO] Sedang melatih model Random Forest...
[INFO] Model berhasil dilatih!


### K-Fold Cross-Validation (K=5)

Selain train-test split biasa, kita juga melakukan **K-Fold 
Cross-Validation dengan K=5** untuk mendapatkan estimasi performa 
model yang lebih robust dan tidak bergantung pada satu pembagian 
data saja. 

Dengan K=5, data training dibagi menjadi 5 lipatan (*fold*). Model 
dilatih sebanyak 5 kali, setiap kali menggunakan 4 fold untuk 
training dan 1 fold untuk validasi. Hasil akhirnya adalah rata-rata 
dari 5 percobaan tersebut.

In [4]:
# K-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    rf_model, X_train, y_train,
    cv=skf,
    scoring=['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'],
    return_train_score=True
)

print("=" * 50)
print("   HASIL K-FOLD CROSS-VALIDATION (K=5)")
print("=" * 50)
print(f"\nAkurasi per Fold    : {[f'{v:.4f}' for v in cv_results['test_accuracy']]}")
print(f"Rata-rata Akurasi   : {cv_results['test_accuracy'].mean():.4f} "
      f"(± {cv_results['test_accuracy'].std():.4f})")
print(f"\nF1-Score (Macro)    : {cv_results['test_f1_macro'].mean():.4f} "
      f"(± {cv_results['test_f1_macro'].std():.4f})")
print(f"Precision (Macro)   : {cv_results['test_precision_macro'].mean():.4f} "
      f"(± {cv_results['test_precision_macro'].std():.4f})")
print(f"Recall (Macro)      : {cv_results['test_recall_macro'].mean():.4f} "
      f"(± {cv_results['test_recall_macro'].std():.4f})")

print("\n[INFO] Perbandingan Train vs Validation Score:")
print(f"Train Accuracy      : {cv_results['train_accuracy'].mean():.4f}")
print(f"Validation Accuracy : {cv_results['test_accuracy'].mean():.4f}")
gap = cv_results['train_accuracy'].mean() - cv_results['test_accuracy'].mean()
print(f"Gap (Train-Val)     : {gap:.4f}")
if gap < 0.05:
    print("[INFO] Gap kecil -> Model tidak overfitting.")
else:
    print("[WARNING] Gap besar -> Model kemungkinan overfitting.")

   HASIL K-FOLD CROSS-VALIDATION (K=5)

Akurasi per Fold    : ['0.9844', '1.0000', '1.0000', '1.0000', '1.0000']
Rata-rata Akurasi   : 0.9969 (± 0.0063)

F1-Score (Macro)    : 0.9947 (± 0.0105)
Precision (Macro)   : 0.9984 (± 0.0033)
Recall (Macro)      : 0.9917 (± 0.0167)

[INFO] Perbandingan Train vs Validation Score:
Train Accuracy      : 1.0000
Validation Accuracy : 0.9969
Gap (Train-Val)     : 0.0031
[INFO] Gap kecil -> Model tidak overfitting.


In [5]:
# Evaluasi pada Test Set
y_pred = rf_model.predict(X_test)

akurasi = accuracy_score(y_test, y_pred)
print("=" * 50)
print("   EVALUASI PADA TEST SET (Hold-out 20%)")
print("=" * 50)
print(f"\nAkurasi Test Set    : {akurasi * 100:.2f}%\n")

target_names = ['Healthy (0)', 'Moderate (1)', 'At Risk (2)']
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

   EVALUASI PADA TEST SET (Hold-out 20%)

Akurasi Test Set    : 100.00%

Classification Report:
              precision    recall  f1-score   support

 Healthy (0)       1.00      1.00      1.00        61
Moderate (1)       1.00      1.00      1.00       149
 At Risk (2)       1.00      1.00      1.00        30

    accuracy                           1.00       240
   macro avg       1.00      1.00      1.00       240
weighted avg       1.00      1.00      1.00       240



In [6]:
# Simpan Model
os.makedirs('../models', exist_ok=True)
model_path = '../models/classifier_model.pkl'
joblib.dump(rf_model, model_path)
print(f"[INFO] Model berhasil disimpan di: {model_path}")

[INFO] Model berhasil disimpan di: ../models/classifier_model.pkl


### Catatan: Akurasi Tinggi pada Dataset Sintetis

Hasil evaluasi menunjukkan akurasi 100% pada test set dan 
rata-rata 99.69% pada K-Fold Cross-Validation (K=5). 

Setelah investigasi melalui Feature Importance, tidak ditemukan 
indikasi data leakage, tidak ada satu fitur pun yang 
mendominasi secara tidak wajar. Fitur yang paling berpengaruh 
adalah perilaku penggunaan layar (*screen behavior*) seperti 
`daily_social_media_hours`, `total_screen_exposure`, dan 
`screen_time_before_sleep` dengan distribusi importance yang 
wajar dan relevan secara domain.

Akurasi tinggi ini disebabkan oleh karakteristik dataset 
`teen_mental_health.csv` yang merupakan **dataset sintetis** 
(data buatan). Dataset sintetis dibuat dengan aturan/formula 
yang deterministik, sehingga pola antara fitur dan target 
bersifat sangat konsisten, berbeda dengan data dunia nyata 
yang penuh noise dan variasi.

Bukti bahwa model **tidak overfitting**:
- Fold 1 pada K-Fold menghasilkan akurasi **98.44%** (tidak 
  sempurna), membuktikan model tidak sekadar menghafal data.
- Gap antara Train Accuracy (100%) dan Validation Accuracy 
  (99.69%) hanya **0.0031**, jauh di bawah threshold 0.05.
- Distribusi Feature Importance wajar dan relevan secara domain.

Meskipun demikian, K-Fold Cross-Validation (K=5) tetap 
dilakukan sebagai validasi yang lebih robust, membuktikan 
bahwa model konsisten dan mampu menggeneralisasi dengan baik 
pada data yang belum pernah dilihat sebelumnya.

Pada tahap ini, model prediksi status kesejahteraan digital remaja 
berbasis **Random Forest Classifier** telah berhasil dibangun, 
divalidasi, dan diekspor.

Ringkasan proses:
1. Data dibagi **80:20** (Training:Testing) dengan stratifikasi kelas.
2. Model divalidasi menggunakan **K-Fold Cross-Validation (K=5)** 
   untuk memastikan performa yang konsisten dan tidak overfitting.
3. Perbandingan *Train Score* vs *Validation Score* digunakan untuk 
   mendeteksi overfitting, gap kecil menandakan model generalisasi 
   dengan baik.
4. Evaluasi akhir dilakukan pada **test set (hold-out 20%)** sebagai 
   simulasi data baru yang belum pernah dilihat model.
5. Model akhir disimpan sebagai `classifier_model.pkl` di folder 
   `/models`, siap diintegrasikan dengan `scaler.pkl` pada tahap 
   **Deployment (Streamlit)**.